In [6]:
# YOLO로 이미지 분류 모델 작성
# YOLO 기반 이미지 분류 모델(ex) YOLO-cls)는 이미지 속 특정 객체의 위치를 찾는 대신,
# 입력된 전체 이미지가 어떤 클래스(카테고리)에 있는지 확인해줌

In [1]:
import random
import shutil
from pathlib import Path
import tensorflow as tf
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns



'''
# 잘못된 캐시 삭제
import os
from pathlib import Path

bad_path = Path.home() / ".keras/datasets/flower_photos"

if bad_path.exists() and not bad_path.is_dir():
    bad_path.unlink()
'''

dataset_path = tf.keras.utils.get_file(
    fname = "flower_photos",
    origin = "http://download.tensorflow.org/example_images/flower_photos.tgz",
    untar=True
)

# 데이터 다운로드 위치
SOURCE_DIR = Path(dataset_path) / "flower_photos"
print("SOURCE_DIR = ", SOURCE_DIR)

# 클래스 목록 보기
classes = [p.name for p in SOURCE_DIR.iterdir() if p.is_dir()]
print("classes : ", classes)

2026-05-27 09:51:40.939237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-27 09:51:40.968477: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-27 09:51:40.968513: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-27 09:51:40.983461: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-27 09:51:42.089142: W tensorflow/compiler/tf

SOURCE_DIR =  /home/pnuav/.keras/datasets/flower_photos/flower_photos
classes :  ['roses', 'dandelion', 'sunflowers', 'daisy', 'tulips']


In [2]:
# YOLO 학습용 dataset 폴더 생성
# flower_dataset이라는 새 데이터셋 폴더 경로를 만든다
DATASET_DIR = Path("flower_dataset")

# 기존 데이터셋 폴더 삭제
# flower_dataset 폴더가 이미 있으면, 안의 내용까지 전부 삭제한다
# 결과: 이전 실행 결과가 남지 않고 새로 만들 수 있다
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

# train / val / test 폴더 경로 설정
# train: 학습용 데이터
# val: 검증용 데이터
# test: 최종 테스트용 데이터
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

# 클래스 폴더 하나씩 확인
# SOURCE_DIR 안에 있는 하위 폴더를 하나씩 가져온다
# 예: daisy, dandelion, roses, sunflowers, tulips
for class_dir in SOURCE_DIR.iterdir():

    # 폴더가 아니면 건너뛰기
    # 이미지 파일이나 기타 파일은 클래스가 아니므로 제외한다
    if not class_dir.is_dir():
        continue
    
    # 클래스 이름 저장
    # 폴더 이름을 클래스 이름으로 사용한다
    # 예: class_dir.name == "roses"
    class_name = class_dir.name

    # 해당 클래스의 모든 이미지 파일 가져오기
    # class_dir 안의 모든 파일을 리스트로 저장한다
    images = list(class_dir.glob("*.*"))

    # 이미지가 없으면 건너뛰기
    # 빈 클래스 폴더는 학습에 사용할 수 없다
    if len(images) == 0:
        continue

    # 이미지 순서 섞기
    # train / val / test가 한쪽으로 치우치지 않도록 랜덤하게 섞는다
    random.shuffle(images)

    # 전체 이미지 개수 저장
    # 현재 클래스에 이미지가 몇 장 있는지 센다
    total = len(images)

    # 전체 이미지 개수 출력
    # 결과: 현재 클래스의 총 이미지 수 확인
    print("total : ", total)

    # train 데이터 끝 위치 계산
    # 전체 이미지 중 70% 지점
    train_end = int(total * 0.7)

    # val 데이터 끝 위치 계산
    # 전체 이미지 중 90% 지점
    # 즉 70%~90% 구간이 val 데이터가 된다
    val_end = int(total * 0.9)

    # 데이터 분할 비율 설정
    # train: 처음 70%
    # val: 70%부터 90%까지, 즉 20%
    # test: 90%부터 끝까지, 즉 10%
    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    # 분할된 이미지 개수 출력
    # 결과: train / val / test에 각각 몇 장 들어갔는지 확인
    print(len(splits["train"]), len(splits["val"]), len(splits["test"]))

    # 분할된 이미지를 각 폴더에 복사
    for split_name, split_images in splits.items():
        # 저장 경로 설정
        target_dir = DATASET_DIR / split_name / class_name
        target_dir.mkdir(parents=True, exist_ok=True)

        for img in split_images:
            shutil.copy2(img, target_dir / img.name)

print()
# dataset 확인 (정상 생성 여부)
for split in ["train", "val", "test"]:
    print(f"[{split}]")

    # 클래스 별 폴더 탐색
    for class_dir in (DATASET_DIR / split).iterdir():
        count = len(list(class_dir.glob("*.*")))
        print(f"{class_dir.name}의 건수는 {count}")
    

total :  641
448 128 65
total :  898
628 180 90
total :  699
489 140 70
total :  633
443 126 64
total :  799
559 160 80

[train]
roses의 건수는 448
dandelion의 건수는 628
sunflowers의 건수는 489
daisy의 건수는 443
tulips의 건수는 559
[val]
roses의 건수는 128
dandelion의 건수는 180
sunflowers의 건수는 140
daisy의 건수는 126
tulips의 건수는 160
[test]
roses의 건수는 65
dandelion의 건수는 90
sunflowers의 건수는 70
daisy의 건수는 64
tulips의 건수는 80


In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
# yolo11classify 데이터로 학습 (train + val 사용)

# 1. 사전 학습 된 YOLO11 classification 모델(coco dataset) 로딩
model = YOLO("yolo11n-cls.pt")

# 2. 모델 학습
model.train(
    data = str(DATASET_DIR.resolve()),    # 경로는 절대 경로 사용 -->> .resolve() : 상대경로를 절대경로로 바꿔줌
    epochs = 5,
    imgsz = 224,
    batch = 16,
    device = 0
)

print("학습 완료")

1.13.1+cu117
True
1
11.7
NVIDIA GeForce MX250
New https://pypi.org/project/ultralytics/8.4.55 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.51 🚀 Python-3.10.20 torch-1.13.1+cu117 CUDA:0 (NVIDIA GeForce MX250, 1994MiB)
WARNING ⚠️ Upgrade to torch>=2.0.0 for deterministic training.
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/pnuav/HyundaiRotem_Bootcamp/python_source/yolo/flower_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, li

In [10]:
# 모델 성능 측정 - 가장 좋은 모델 로딩
best_model = YOLO("runs/classify/train-2/weights/best.pt")

y_true = []         # 실제 정답
y_pred = []         # 예측값

test_dir = DATASET_DIR / "test"

for class_dir in test_dir.iterdir():
    true_label = class_dir.name

    for img_path in class_dir.glob("*.*"):
        # 예측 수행
        results = best_model.predict(source=str(img_path),imgsz=224,verbose=False)

        # 결과 1개 추출
        r = results[0]

        # 가장 높은 확률의 class
        pred_idx = r.probs.top1
        pred_label = r.names[pred_idx]

        # 실제값 기록
        y_true.append(true_label)
        y_pred.append(pred_label)


# 성능 출력
acc = accuracy_score(y_true=y_true, y_pred=y_pred)
print(f"test acc : {acc * 100:.2f}%")
print()
print(classification_report(y_true=y_true, y_pred=y_pred))
print()

# confusion matrix 시각화 (heatmap)
cm = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=classes)
plt.figure(figsize=(8, 6))
sns.heatmap(data=cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, cmap="Blues")
plt.title("Confusion Matrix [test data]")
plt.xlabel("Predicted")
plt.ylabel("Truth value")
plt.tight_layout()
plt.savefig("yolo11_confusion_matrix[test data].png")
plt.show()


test acc : 90.79%

              precision    recall  f1-score   support

       daisy       0.95      0.94      0.94        64
   dandelion       0.88      0.94      0.91        90
       roses       0.87      0.91      0.89        65
  sunflowers       0.97      0.93      0.95        70
      tulips       0.89      0.82      0.86        80

    accuracy                           0.91       369
   macro avg       0.91      0.91      0.91       369
weighted avg       0.91      0.91      0.91       369




<Figure size 800x600 with 2 Axes>

In [12]:
# 꽃 이미지 분류
print("다운받은 꽃 이미지 검증 성능 확인")

sample_path = Path("myflower.jpeg")

if sample_path.exists():
    results = best_model.predict(source=str(sample_path), imgsz=224)

    # 첫 번째 예측 결과 가져오기
    r = results[0]

    # r.probs : 분류 모델의 예측 확률 정보
    # - 이미지가 각 클래스일 확률을 담고 있음
    # - 예: roses 0.80, tulips 0.10, daisy 0.05 ...

    pred_idx = r.probs.top1
    # 가장 확률이 높은 클래스의 번호(index)

    pred_label = r.names[pred_idx]
    # 클래스 번호를 실제 클래스 이름으로 변환

    confidence = float(r.probs.top1conf)
    # 가장 높게 예측한 클래스의 확률값
    # 즉, 모델이 pred_label이라고 확신한 정도

    print(sample_path)
    print("Predictd class : ", pred_label)
    print("confidence : ", round(confidence, 3))
else:
    print("파일 없음")

다운받은 꽃 이미지 검증 성능 확인
myflower.jpeg
Predictd class :  roses
confidence :  0.999
